# Reward Modeling on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/llm-post-training/blob/main/notebooks/02_reward_modeling_colab.ipynb)

This notebook demonstrates reward modeling for RLHF on Google Colab with GPU acceleration.

**What you'll learn:**
- Training reward models from preference data
- Bradley-Terry ranking loss
- Evaluating ranking accuracy
- Using GPU for faster training

**Estimated time:** 15-30 minutes

## Setup: Clone Repository and Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone repository (update with your repo URL)
!git clone https://github.com/yourusername/llm-post-training.git
%cd llm-post-training

In [ ]:
# Install dependencies with GPU support
!pip install -q -e ".[gpu,experiment]"

print("\n✅ Installation complete!")

In [ ]:
# Verify installation
import torch
from transformers import __version__ as transformers_version

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers_version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## Quick Test: Train on Synthetic Data

Let's start with a quick test on synthetic data to verify everything works.

In [ ]:
# Quick training run (1K examples, 3 epochs, ~5 minutes)
!python scripts/train/train_reward_model.py \
  experiment=reward_gpt2_synthetic \
  training.num_epochs=3 \
  training.per_device_train_batch_size=8 \
  training.fp16=true \
  data.num_train_examples=1000 \
  data.num_eval_examples=100

**Expected Results:**
- Training time: ~5 minutes on T4 GPU
- Final accuracy: 75-85% (synthetic data has clear patterns)
- Reward margin: > 0.5 (good separation)

If you see accuracy > 70%, the reward model is working! ✅

## Real Training: Anthropic HH-RLHF Dataset

Now let's train on real human preference data from Anthropic.

In [ ]:
# Train on Anthropic HH-RLHF (10K examples, ~30 minutes)
!python scripts/train/train_reward_model.py \
  experiment=reward_gpt2_hh_rlhf \
  training.fp16=true \
  training.per_device_train_batch_size=8 \
  data.num_train_examples=10000 \
  data.num_eval_examples=1000

**Expected Results:**
- Training time: ~30 minutes on T4 GPU
- Final accuracy: 65-75% (real data is noisier)
- Ready for RLHF!

**Note:** Real human preferences are more ambiguous than synthetic data, so accuracy will be lower.

## 🔧 Troubleshooting Poor Performance

**Got accuracy < 60%? Here's how to fix it:**

### Quick Fixes (Try These First)

#### 1. 🎯 Lower Learning Rate (Most Common Fix)
Learning rate 1e-5 is often too high for reward modeling. Try 2e-6 or 1e-6:

```python
!python scripts/train/train_reward_model.py \
  experiment=reward_gpt2_hh_rlhf \
  training.fp16=true \
  training.learning_rate=2e-6 \
  training.per_device_train_batch_size=8 \
  data.num_train_examples=10000
```

#### 2. 🔄 Train Longer
1 epoch might not be enough. Try 2-3 epochs:

```python
!python scripts/train/train_reward_model.py \
  experiment=reward_gpt2_hh_rlhf \
  training.fp16=true \
  training.num_epochs=3 \
  training.per_device_train_batch_size=8 \
  data.num_train_examples=10000
```

#### 3. 📊 Use More Data
10K examples might be insufficient. Try 30K-50K:

```python
!python scripts/train/train_reward_model.py \
  experiment=reward_gpt2_hh_rlhf \
  training.fp16=true \
  training.per_device_train_batch_size=8 \
  data.num_train_examples=50000 \
  data.num_eval_examples=2000
```

### Advanced Fixes

#### 4. Try Without LoRA
Disable LoRA to see if it's causing issues:

```python
!python scripts/train/train_reward_model.py \
  experiment=reward_gpt2_hh_rlhf \
  training.fp16=true \
  training.per_device_train_batch_size=4 \
  model.use_lora=false \
  data.num_train_examples=10000
```

#### 5. Increase LoRA Rank
If using LoRA, try larger rank (r=32 or r=64):

```python
!python scripts/train/train_reward_model.py \
  experiment=reward_gpt2_hh_rlhf \
  training.fp16=true \
  training.per_device_train_batch_size=8 \
  model.lora_config.r=32 \
  model.lora_config.lora_alpha=64 \
  data.num_train_examples=10000
```

### 🌟 Optimal Configuration (Recommended)

This combines best practices for Anthropic HH-RLHF:

```python
!python scripts/train/train_reward_model.py \
  experiment=reward_gpt2_hh_rlhf \
  training.fp16=true \
  training.learning_rate=2e-6 \
  training.num_epochs=2 \
  training.per_device_train_batch_size=8 \
  training.gradient_accumulation_steps=2 \
  model.lora_config.r=32 \
  model.lora_config.lora_alpha=64 \
  data.num_train_examples=30000 \
  data.num_eval_examples=2000
```

**Expected results:** 65-72% accuracy, 20-30 minutes on T4

### Troubleshooting Guide

| Symptom | Likely Cause | Solution |
|---------|--------------|----------|
| Accuracy 50-56% | LR too high, not learning | Lower LR to 2e-6 or 1e-6 |
| Loss not decreasing | Model not training | Check data format, lower LR |
| Accuracy plateaus early | Insufficient training | Increase epochs to 2-3 |
| Margin < 0.01 | Weak discrimination | More data + lower LR |
| Accuracy drops after peak | Overfitting | Reduce LR, early stopping |
| Same reward for all inputs | Training collapsed | Restart with LR=1e-6 |

### Quick Diagnostic

Run this to verify your setup works:

```python
# Should achieve 75-85% accuracy on synthetic data
!python scripts/train/train_reward_model.py \
  experiment=reward_gpt2_synthetic \
  training.num_epochs=3 \
  training.fp16=true \
  training.per_device_train_batch_size=8 \
  data.num_train_examples=2000
```

**If synthetic works but HH-RLHF doesn't:**
- Problem is with real data training → Use solutions above
- Most likely: learning rate too high

**If synthetic also fails:**
- Problem with basic setup → Check GPU, reinstall dependencies

## Interactive Exploration

Let's load the trained model and test it interactively.

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
from src.models.reward import RewardModel

# Load the trained reward model
print("Loading reward model...")
reward_model = RewardModel.from_pretrained("./outputs/reward_gpt2_synthetic/final_model")
reward_model.eval()

print(f"✓ Reward model loaded on {reward_model.device}")

In [ ]:
# Test the reward model on custom examples
def score_response(prompt, response):
    """Score a prompt-response pair."""
    text = prompt + response
    tokens = reward_model.tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    )
    tokens = {k: v.to(reward_model.device) for k, v in tokens.items()}
    
    with torch.no_grad():
        reward = reward_model(
            input_ids=tokens['input_ids'],
            attention_mask=tokens['attention_mask'],
            return_dict=False,
        )
    
    return reward.item()

# Test examples
test_cases = [
    {
        "prompt": "What is 15 + 7?",
        "good": "15 + 7 equals 22.",
        "bad": "I don't know.",
    },
    {
        "prompt": "What is the capital of France?",
        "good": "The capital of France is Paris.",
        "bad": "France has a capital city.",
    },
    {
        "prompt": "Explain photosynthesis.",
        "good": "Photosynthesis is the process by which plants convert light energy into chemical energy.",
        "bad": "Plants do photosynthesis.",
    },
]

print("\nTesting Reward Model:")
print("=" * 80)

for i, test in enumerate(test_cases):
    reward_good = score_response(test['prompt'], test['good'])
    reward_bad = score_response(test['prompt'], test['bad'])
    margin = reward_good - reward_bad
    
    print(f"\n[Test {i+1}]")
    print(f"Prompt: {test['prompt']}")
    print(f"\n  ✅ Good: {test['good']}")
    print(f"     Reward: {reward_good:.4f}")
    print(f"\n  ❌ Bad: {test['bad']}")
    print(f"     Reward: {reward_bad:.4f}")
    print(f"\n  📊 Margin: {margin:.4f} {'✓ Correct!' if margin > 0 else '✗ Wrong'}")
    print("-" * 80)

## Understanding the Results

### Key Metrics:

**Ranking Accuracy:**
- 50%: Random guessing (model learned nothing)
- 60-65%: Weak signal (needs improvement)
- 70-80%: Good performance (ready for RLHF)
- 85%+: Excellent (very reliable)

**Reward Margin:**
- < 0.1: Weak separation (low confidence)
- 0.1-0.5: Moderate separation
- > 0.5: Strong separation (confident rankings)

### What Makes a Good Reward Model:

1. ✅ **High accuracy** (> 70%): Reliably ranks preferences
2. ✅ **Large margin**: Confident about good vs bad
3. ✅ **Calibrated**: Confidence matches actual accuracy
4. ✅ **Generalizes**: Works on unseen examples

## Visualize Training Dynamics

Let's plot how the model improved during training.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import glob

# Load training logs from TensorBoard
from tensorboard.backend.event_processing import event_accumulator

log_dir = "./outputs/reward_gpt2_synthetic/logs"
event_files = glob.glob(f"{log_dir}/events.out.tfevents.*")

if event_files:
    ea = event_accumulator.EventAccumulator(log_dir)
    ea.Reload()
    
    # Extract metrics
    metrics_to_plot = [
        ('train/loss', 'Training Loss'),
        ('train/accuracy', 'Training Accuracy'),
        ('train/reward_margin', 'Reward Margin'),
    ]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, (metric_name, title) in enumerate(metrics_to_plot):
        if metric_name in ea.Tags()['scalars']:
            events = ea.Scalars(metric_name)
            steps = [e.step for e in events]
            values = [e.value for e in events]
            
            axes[idx].plot(steps, values, linewidth=2)
            axes[idx].set_title(title, fontsize=14, fontweight='bold')
            axes[idx].set_xlabel('Step')
            axes[idx].set_ylabel(title)
            axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No training logs found. Run training first!")

## Save Model to Google Drive

Save the trained model to Google Drive for later use.

In [ ]:
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Copy model to Drive
drive_path = '/content/drive/MyDrive/llm_models/reward_model'
!mkdir -p {drive_path}

shutil.copytree(
    './outputs/reward_gpt2_synthetic/final_model',
    drive_path,
    dirs_exist_ok=True
)

print(f"✅ Model saved to: {drive_path}")

## Next Steps

Now that you have a trained reward model, you can:

1. **Phase 4: DPO** - Train policy directly from preferences (simpler than RLHF)
2. **Phase 5: PPO/RLHF** - Use this reward model to optimize a policy with RL
3. **Experiment** - Try different base models, datasets, and hyperparameters

### Recommended Experiments:

**Different base models:**
```python
!python scripts/train/train_reward_model.py \
  model=opt-350m \
  experiment=reward_gpt2_hh_rlhf
```

**More training data:**
```python
!python scripts/train/train_reward_model.py \
  experiment=reward_gpt2_hh_rlhf \
  data.num_train_examples=50000  # Use more data
```

**Frozen base model (faster):**
```python
!python scripts/train/train_reward_model.py \
  experiment=reward_gpt2_synthetic \
  technique.freeze_base_model=true  # Only train value head
```

---

## Summary

In this notebook, you:
- ✅ Trained a reward model on synthetic data (80%+ accuracy)
- ✅ Trained on real human preferences (Anthropic HH-RLHF)
- ✅ Tested the model interactively
- ✅ Understood key metrics (accuracy, margin)
- ✅ Saved the model for RLHF

**Congratulations!** You've completed Phase 3: Reward Modeling! 🎉

Ready for Phase 4 (DPO) or Phase 5 (PPO/RLHF)?